# Chapter 4: Validation of correlation and detecting of noise
## Create AAV6-ML figures for Chapter 3 for report for internship in AG Grimm
Author: Kolja Hildenbrand

Created: 22/04/2026

Finished: ...

## 1. Packages

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import re
import matplotlib as plt
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from itertools import product
from matplotlib_venn import venn2
from scipy.stats import pearsonr, spearmanr, linregress
from matplotlib.patches import Patch
import math


## 2. Preperation 
### 2.1. Create Paths

In [ ]:
#---------------------------------
#---------------------------------
#---------------------------------

#general Path

general_dir = Path('/Users/kollybook/Library/Mobile Documents/com~apple~CloudDocs/Kolja_Hildenbrand/Uni/Master_Infectious_Diseases/Grimm_internship/Report_Grimm/Data') # <----- Hier den Server Path anpassen
os.makedirs(general_dir, exist_ok=True)

#---------------------------------
#---------------------------------
#---------------------------------

# Path for NGS data
NGS_dir = general_dir/'NGS_data'
os.makedirs(NGS_dir, exist_ok=True)

# Path for tables
tables_dir = general_dir/'tables'
os.makedirs(tables_dir, exist_ok=True)

#Path for figures
figures_dir = general_dir/'figures'/'6_validation_noise'
os.makedirs(figures_dir, exist_ok=True)

### 2.2. Define helper functions


In [ ]:
# read csv files
def read_csv_file (path, file_name):
    df = pd.read_csv(path / f"{file_name}.csv")
    return df

In [ ]:
# extract list information from table
def extract_info(table, *columns):
    results = []
    
    for col in columns:
        if col not in table.columns:
            raise ValueError(f"Column '{col}' not found in table")
        
        unique_vals = (
            table[col]
            .dropna()
            .unique()
            .tolist()
        )
        
        results.append(sorted(unique_vals))
    
    return tuple(results)

In [ ]:
# sort file names from df_long
def sort_file_names (name_list):
    FILE_NAMES = {
        "gDNA": {
            "heart": {"biological": [], "technical": []},
            "liver": {"biological": [], "technical": []}
        },
        "cDNA": {
            "heart": {"biological": [], "technical": []},
            "liver": {"biological": [], "technical": []}
        }
    }
    
    for name in name_list:
        parts = name.split("_")
        n_parts = len(parts)
    
        # ext type
        if name.startswith("gDNA"):
            dna = "gDNA"
        elif name.startswith("cDNA"):
            dna = "cDNA"
        else:
            continue
    
        # Tissue 
        tissue = parts[1]
    
        if tissue not in ["heart", "liver"]:
            continue  
    
        # bio or tech
        if n_parts == 3:
            category = "biological"
        elif n_parts == 4:
            category = "technical"
        else:
            continue
    
        FILE_NAMES[dna][tissue][category].append(name)

    return FILE_NAMES 

## 3. Script Functions

### 3.1. Leave one out scatterplot

In [ ]:
def get_samples(table, tissue, extraction, single_mouse_ID='f2', new=False):
    # ----------------------------------------------------------
    # Select biological replicate data for one tissue/extraction
    # ----------------------------------------------------------
    df = table[
        (table['Replicate'] == 'biological') &
        (table['Tissue'] == tissue) &
        (table['Extraction_type'] == extraction)
    ][['AA_sequence', 'Mouse_ID', 'Log2_enrichment', 'RPM', 'RPM_input']].copy()

    # ----------------------------------------------------------
    # Select single mouse that will be compared against pooled others
    # ----------------------------------------------------------
    df_one = df[df['Mouse_ID'] == single_mouse_ID][['AA_sequence', 'Log2_enrichment']].copy()

    # ----------------------------------------------------------
    # Exclude selected mouse from pooled group
    # ----------------------------------------------------------
    df_pool = df[df['Mouse_ID'] != single_mouse_ID].copy()

    # ----------------------------------------------------------
    # Pool remaining mice across AA_sequence
    # ----------------------------------------------------------
    df_pool_grouped = df_pool.groupby("AA_sequence", as_index=False).agg({
        "Log2_enrichment": "mean",
        "RPM": "mean",
        "RPM_input": "first"
    })

    # ----------------------------------------------------------
    # Optional: calculate new Log2_enrichment from pooled RPM / RPM_input
    # ----------------------------------------------------------
    if new:
        df_pool_grouped['Log2_enrichment'] = np.log2(
            df_pool_grouped['RPM'] / df_pool_grouped['RPM_input']
        )

    # ----------------------------------------------------------
    # Keep only columns needed for correlation / scatter plot
    # ----------------------------------------------------------
    df_pool_grouped = df_pool_grouped[['AA_sequence', 'Log2_enrichment']].copy()

    return df_one, df_pool_grouped

In [ ]:
# Function for scatterplot
def corr_plot_two_samples(
    df1,
    df2,
    sample_1,
    sample_2,
    value_col="Log2_enrichment",
    aa_col="AA_sequence",
    title=True,
    figsize=(5, 5),
    alpha=0.2,
    point_size=5,
    cmap="viridis",
    save=False,
    output_path=None,
    file_name=None
):
    """
    Plot the correlation between two samples / groups after merging on AA_sequence.

    This function is agnostic to whether the inputs represent:
    - biological replicates
    - technical replicates
    - pooled sex groups
    - any other pre-filtered sample tables

    Parameters
    ----------
    df1, df2 : pd.DataFrame
        Input tables for the two samples/groups to compare.
        Each must contain at least:
        [aa_col, value_col, pseudo_col]
    sample_1, sample_2 : str
        Names used for axis labels and plot title.
    value_col : str, default='Log2_enrichment'
        Numeric column to correlate.
    aa_col : str, default='AA_sequence'
        Column used to merge the two tables.
    title : bool, default=True
        Whether to show a title.
    figsize : tuple, default=(5, 5)
        Figure size.
    alpha : float, default=0.2
        Transparency for scatter points.
    point_size : int, default=5
        Point size for scatter plot.
    cmap : str, default='viridis'
        Colormap for hexbin.
    save : bool, default=False
        Whether to save the figure.
    output_path : str or Path, optional
        Save directory. Required if save=True.
    file_name : str, optional
        Output file name.

    Returns
    -------
    df_merge : pd.DataFrame
        Merged and filtered table used for plotting.
    r : float
        Pearson correlation coefficient.
    """

    # ----------------------------------------------------------
    # Keep only required columns and rename for merge
    # ----------------------------------------------------------
    d1 = df1[[aa_col, value_col]].copy()
    d2 = df2[[aa_col, value_col]].copy()
        
    d1 = d1.rename(columns={
        value_col: f"{value_col}_{sample_1}"
    })

    d2 = d2.rename(columns={
        value_col: f"{value_col}_{sample_2}",
    })

    # ----------------------------------------------------------
    # Merge on AA sequence
    # ----------------------------------------------------------
    df_merge = d1.merge(d2, on=aa_col, how="inner")

    x_col = f"{value_col}_{sample_1}"
    y_col = f"{value_col}_{sample_2}"

    # ----------------------------------------------------------
    # Create figure
    # ----------------------------------------------------------
    fig, ax = plt.subplots(figsize=figsize)

    ax.scatter(
        df_merge[x_col],
        df_merge[y_col],
        alpha=alpha,
        s=point_size
    )

    # ----------------------------------------------------------
    # Pearson correlation
    # ----------------------------------------------------------
    r, _ = pearsonr(df_merge[x_col], df_merge[y_col])

    ax.text(
        0.05, 0.95,
        f"Pearsonr = {r:.2f}",
        transform=ax.transAxes,
        fontsize=12,
        verticalalignment="top"
    )

    # ----------------------------------------------------------
    # Diagonal line
    # ----------------------------------------------------------
    lims = [
        min(df_merge[x_col].min(), df_merge[y_col].min()),
        max(df_merge[x_col].max(), df_merge[y_col].max())
    ]
    ax.plot(lims, lims, "--", color="black")
    
    # ----------------------------------------------------------
    # Regression line
    # ----------------------------------------------------------
    x = df_merge[x_col]
    y = df_merge[y_col]

    res = linregress(x, y)
    x_vals = np.linspace(x.min(), x.max(), 100)

    ax.plot(
        x_vals,
        res.slope * x_vals + res.intercept,
        color="red",
        linewidth=2
    )

    # ----------------------------------------------------------
    # Labels and title
    # ----------------------------------------------------------
    ax.set_xlabel(f"{value_col.replace('_', ' ')} ({sample_1})")
    ax.set_ylabel(f"{value_col.replace('_', ' ')} ({sample_2})")

    if title:
        ax.set_title(f"Correlation between {sample_1} and {sample_2}")

    plt.tight_layout()

    # ----------------------------------------------------------
    # Save
    # ----------------------------------------------------------
    if save:
        if output_path is None:
            raise ValueError("Please provide output_path when save=True")
        os.makedirs(output_path, exist_ok=True)

        if file_name is None:
            file_name = f"scatter_corr_{sample_1}_vs_{sample_2}.png"

        fig.savefig(os.path.join(output_path, file_name), dpi=300, bbox_inches="tight")

    plt.show()

    return df_merge, r

In [ ]:
def store_corr_value_df(corr_table, value, tissue, extraction, column, corr_style):
    """
    Append a single correlation value to a long-format correlation table.
    """

    df_new = pd.DataFrame({
        "Tissue": [tissue],
        "Extraction_type": [extraction],
        "Comparison": [column],
        "Correlation_style": [corr_style],
        "Correlation": [value]
    })

    corr_table = pd.concat([corr_table, df_new], ignore_index=True)

    return corr_table

In [ ]:
# combine get sample and create plot
def leave_one_out (single_mouse_ID, ext, tissue, new=False, title=True, add_to_table=False, corr_table = None, save=False, output_path=None):
    
    single, pooled = get_samples(
        table=df_long, 
        tissue=tissue, 
        extraction=ext, 
        single_mouse_ID=single_mouse_ID,
        new=new)
    
    df_merge, r = corr_plot_two_samples(
        df1=single, 
        df2=pooled,
        sample_1=f"{ext} {tissue} {single_mouse_ID}",
        sample_2= f"{ext} {tissue} without {single_mouse_ID} pooled",
        value_col="Log2_enrichment", 
        title=title,
        save=save,
        output_path=output_path
    )

    if add_to_table:
         corr_table = store_corr_value_df(
             corr_table=corr_table, 
             value=r, 
             tissue = tissue, 
             extraction = ext, 
             column = 'leave_one_out', 
             corr_style = 'pearson'
         )
    return corr_table

In [ ]:
df

### 3.2. Noise vs Abundance scatter (log - log)

## 4. Load files and extract information

### 4.1. Load csv files

In [ ]:
%%time
#load long table
df_long = read_csv_file(tables_dir / 'summary', "df_long_combined_biological_technical_rep")

# load pooled table
df_pooled = read_csv_file(tables_dir / 'summary', "df_sample_pooled")

# load pooled sex table
df_pooled_sex = read_csv_file(tables_dir / 'summary', "df_sample_pooled_sex")

In [ ]:
print ('Long Table')
display (df_long)

print ('Pooled Table')
display (df_pooled)

print ('Pooled Table sex')
display (df_pooled_sex)

In [ ]:
# Load correlation values from bio, tech, sex from chapter 4
df_corr_values = read_csv_file(tables_dir/ 'Correlation_table', "correlation_heatmaps_bio_tech_sex")
df_corr_values = df_corr_values[df_corr_values['Comparison']== 'Mouse_ID']

In [ ]:
df_corr_values.reset_index(drop=True)

### 4.2. Extract information from table

In [ ]:
SAMPLE, EXT, TISSUE, SEX, MOUSE_ID = extract_info(
    df_long, 
    'Sample', 
    'Extraction_type',
    "Tissue", 
    'Sex', 
    'Mouse_ID'
)

In [ ]:
display (SAMPLE, EXT, TISSUE, SEX, MOUSE_ID)

In [ ]:
# Sort different file names in extraction and biological or technical Replicates
DICT_NAMES = sort_file_names (SAMPLE)
DICT_NAMES

### 4.3. Load tissue/ext specific librarys¶

In [ ]:
%%time
# Load tissue specific library 
dict_library = {}
for tissue in TISSUE:
    df = read_csv_file(tables_dir/tissue, f'df_{tissue}_input_library')
    dict_library[tissue] = df

# Load raw library  for corr with special library
df_raw_input =  read_csv_file(tables_dir, 'df_input_lib_raw')

## 5. Figures

### 5.1. Leave one out scatterplot

In [ ]:
for ext, tissue, mouse in product (EXT, TISSUE, MOUSE_ID):
    df_corr_values = leave_one_out (single_mouse_ID=mouse, ext=ext, tissue=tissue, new=False, title=False, add_to_table=True, corr_table=df_corr_values, save=True, output_path=figures_dir/'1_leave_one_out'/'all_scatter_plots')
    display (df_corr_values)
    

In [ ]:
for ext, tis in product (EXT, TISSUE):
    leave_one_out (single_mouse_ID='f2', ext=ext, tissue=tis, new=False, title=False, table=False, save=True, output_path=figures_dir/'1_leave_one_out')

In [ ]:
for ext, tis in product (EXT, TISSUE):
    leave_one_out (single_mouse_ID='f2', ext=ext, tissue=tis, new=True, title=False, save=True, output_path=figures_dir/'1_leave_one_out'/'log2(mean)_calculation')

In [ ]:
def prepare_corr_plot_table(corr_table, corr_style="pearson"):
    """
    Prepare correlation table for plotting biological vs leave-one-out comparisons.
    """
    df = corr_table[corr_table["Correlation_style"] == corr_style].copy()
    # combine extraction + tissue for x-axis
    df["Sample_type"] = df["Extraction_type"] + "_" + df["Tissue"]
    # prettier x labels
    sample_label_map = {
        "gDNA_liver": "gDNA\nLiver",
        "cDNA_liver": "cDNA\nLiver",
        "gDNA_heart": "gDNA\nHeart",
        "cDNA_heart": "cDNA\nHeart",
    }

    df["Sample_label"] = df["Sample_type"].map(sample_label_map)
  
    # prettier comparison labels
    comparison_label_map = {
        "Mouse_ID": "Biological",
        "leave_one_out": "Leave-one-out"
    }

    df["Comparison_label"] = df["Comparison"].map(comparison_label_map)

    return df

In [ ]:
def plot_biological_vs_leaveoneout(
    corr_table,
    corr_style="pearson",
    sample_order=None,
    hue_order=("Biological", "Leave-one-out"),
    figsize=(10, 6),
    title=True,
    save=False,
    output_path=None,
    file_name=None
):
    """
    Plot biological pairwise correlations vs leave-one-out correlations
    across all sample types.
    """

    df = prepare_corr_plot_table(corr_table, corr_style=corr_style)

    # keep only the two relevant comparison groups
    df = df[df["Comparison"].isin(["Mouse_ID", "leave_one_out"])].copy()

    if sample_order is None:
        sample_order = ["gDNA\nLiver", "cDNA\nLiver", "gDNA\nHeart", "cDNA\nHeart"]

    palette = {
        "Biological": "#4C78A8",
        "Leave-one-out": "#E45756"
    }

    sns.set_style("whitegrid")
    plt.rcParams.update({
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.fontsize": 10
    })

    fig, ax = plt.subplots(figsize=figsize)

    # boxplot
    sns.boxplot(
        data=df,
        x="Sample_label",
        y="Correlation",
        hue="Comparison_label",
        order=sample_order,
        hue_order=hue_order,
        palette=palette,
        showfliers=False,
        width=0.8,
        ax=ax
    )

    # individual points
    sns.stripplot(
        data=df,
        x="Sample_label",
        y="Correlation",
        hue="Comparison_label",
        order=sample_order,
        hue_order=hue_order,
        dodge=True,
        palette=palette,
        edgecolor='black',
        alpha=1,
        size=4,
        linewidth=0.2,
        ax=ax
    )

    # fix duplicated legend (boxplot + stripplot both add one)
    handles, labels = ax.get_legend_handles_labels()
    unique = dict(zip(labels[:len(hue_order)], handles[:len(hue_order)]))
    ax.legend(
        unique.values(),
        unique.keys(),
        title="Comparison",
        frameon=False,
        loc="lower right"
    )

    ax.set_xlabel("")
    ax.set_ylabel(f"{corr_style.capitalize()} correlation")
    ax.set_ylim(0, 1)

    if title:
        ax.set_title(f"Biological vs leave-one-out replicate correlations ({corr_style})")

    ax.grid(axis="y", linestyle="--", linewidth=0.7, alpha=0.6)
    ax.grid(axis="x", visible=False)
    sns.despine(ax=ax)

    plt.tight_layout()

    if save:
        if output_path is None:
            raise ValueError("Please provide output_path when save=True")
        os.makedirs(output_path, exist_ok=True)

        if file_name is None:
            file_name = f"biological_vs_leaveoneout_{corr_style}.png"

        fig.savefig(os.path.join(output_path, file_name), dpi=300, bbox_inches="tight")

    plt.show()

In [ ]:
df_corr_values

In [ ]:
plot_biological_vs_leaveoneout(
    corr_table = df_corr_values,
    corr_style="pearson",
    figsize=(6,6),
    title=False,
    save=True,
    output_path=figures_dir/'2_bio_vs_leave_1_out'
)